# P84 — Bosque de aislamiento

## 1. Título y paper

**Paper:** *Isolation Forest*  
**Autoría:** Fei Tony Liu, Kai Ming Ting, Zhi-Hua Zhou  
**Año y venue:** 2008 · ICDM 2008, 413–422  
**Nivel:** L2 · **Motor:** `isolation_forest`  
**Ficha completa:** [`P84_isolation_forest`](../../papers/foundational/P84_isolation_forest/README.md)

**Hito:** Invierte el planteamiento de la detección de anomalías: en vez de modelar lo normal, mide lo fácil que es aislar cada punto.

- [doi:10.1109/ICDM.2008.17](https://doi.org/10.1109/ICDM.2008.17)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los métodos de detección de anomalías construían un modelo de la normalidad y medían la distancia a él. Eso cuesta caro, supone una forma para la distribución normal y dedica casi todo el esfuerzo a los puntos que no interesan.
2. Ejecutar una implementación mínima de la propuesta: Cortar el espacio al azar y contar cuántos cortes hacen falta para dejar cada punto solo. Lo raro vive en zonas poco pobladas y se aísla antes; la longitud media del camino, normalizada, es la puntuación de anomalía.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Breunig et al. (2000), factor de anomalía local
- P79


## 4. Intuición

Todos los métodos de detección de anomalías modelaban primero qué es normal. Este da la vuelta a la pregunta: corta el espacio al azar y cuenta cuántos cortes hacen falta para dejar cada punto solo. Lo raro está donde hay poca gente, y se queda solo enseguida.


## 5. Concepto mínimo

```text
h(x) = número de cortes aleatorios para aislar x

Puntuación:  s(x) = 2^(−E[h(x)] / c(m))
    c(m) = longitud media de camino en un árbol de búsqueda binaria con m nodos

s → 1  : se aísla enseguida     → anomalía
s → 0,5: camino medio            → normal
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('isolation_forest', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿En cuántos cortes se aísla un punto anómalo?
2. ¿Y uno normal?
3. ¿Quedan las anomalías en las primeras posiciones del ranking?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('isolation_forest', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('isolation_forest', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Los anómalos se aíslan en **2,27 cortes** de media y los normales en **6,64**. La puntuación traduce eso a 0,772 frente a 0,471. Y las **3 de 3** anomalías reales quedan en las tres primeras posiciones del ranking.


## 10. Comentario pedagógico

La inversión conceptual es lo valioso: no hay que suponer ninguna forma para la distribución normal, no hay que calcular distancias entre todos los pares, y el coste es lineal. El método detecta bien anomalías **globales** —puntos alejados de todo— y mal las **locales**, que viven dentro de la nube con densidad distinta.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar un bosque de aislamiento para anomalías locales.


In [ ]:
print('Un punto dentro de la nube pero en una zona de densidad distinta')
print('NO se aisla antes que sus vecinos: este metodo no lo va a ver.')
print('Para eso existe el factor de anomalia local (LOF), con otra idea.')

## 12. Corrección

Lo que sí detecta bien, con los números delante:


In [ ]:
r = run_paper_lab('isolation_forest', seed=7)['result']
print('camino medio normales :', r['longitud_media_de_camino_normales'])
print('camino medio anomalos :', r['longitud_media_de_camino_anomalos'])
for fila in r['top_5_del_ranking']:
    print('  ', fila)

## 13. Desafío guiado

Comprueba en el top del ranking cuántas de las tres anomalías reales aparecen, y qué puntuación separa a las anomalías del resto.


In [ ]:
r = run_paper_lab('isolation_forest', seed=3)['result']
show(r)

## 14. Desafío autónomo

Aplica un bosque de aislamiento a un registro real de eventos, ajusta la proporción esperada de anomalías y revisa a mano las veinte primeras. Documenta cuántas eran anomalías de verdad.


## 15. Evidencia de aprendizaje

Guarda la comparación de longitudes de camino y tu criterio para elegir entre detección global y local.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P84_isolation_forest/README.md) · evaluación formal: [`assessments/papers/P84_isolation_forest.md`](../../assessments/papers/P84_isolation_forest.md)


## 16. Cierre

Los datos ya se agrupan, se visualizan y se auditan. Queda el problema de predecir cuando la mayor parte de la tabla está vacía.


## 17. Conexión con el siguiente hito

- P42

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
